---
date: "2026-04-15"
date-modified: last-modified
format:
  html:
    toc: true
---


# Simpson's Paradox

## Definition and Core Concept
**Simpson's Paradox** is a phenomenon in probability and statistics where a clear trend appears in several different groups of data but disappears or reverses when those groups are combined. 

It is important to note that this is not a "true" paradox in the sense of a logical mathematical contradiction, but rather a result that is deeply counterintuitive to human perception. It proves that the **sign of inequalities can definitively flip** when data is aggregated, highlighting the dangers of analyzing unstratified data.



---

## The Crux: The Hidden Confounder

A common statistical stumbling block is assuming that if condition A is strictly better than condition B in all individual sub-scenarios, it must be better overall. This ignores the varying sizes and hidden complexities of the subsets.

::: {.callout-important}
## Shifting Weights and Base Rates
The paradox occurs because of a **confounding variable**—a hidden factor that influences both the independent and dependent variables. When data is aggregated, the varying base rates (or sample sizes) of the subgroups dominate the calculation. If a "poor" performer takes on the vast majority of "easy" cases, their overall success rate is artificially inflated compared to a "top" performer who handles a smaller volume of highly difficult cases. 
:::

---

## The "Two Surgeons" Example (Intuition)
To illustrate why this happens, consider two surgeons: **Doctor A** (a highly skilled specialist) and **Doctor B** (a general practitioner). We compare their success rates across two types of procedures: Heart Surgery (Hard) and Band-Aid Removal (Easy).

| Surgery Type | Doctor A (Success Rate) | Doctor B (Success Rate) |
| :--- | :--- | :--- |
| **Heart Surgery (Hard)** | **70/90 (77.8%)** | 2/10 (20.0%) |
| **Band-Aid Removal (Easy)** | **10/10 (100%)** | 81/90 (90.0%) |
| **Total (Aggregated)** | 80/100 (80.0%) | **83/100 (83.0%)** |

### Why the flip occurs:

* **Separately (Conditioned):** Doctor A is better at heart surgery *and* better at band-aid removal.
* **Aggregated (Unconditioned):** Doctor B appears to be the superior surgeon with an $83\%$ success rate compared to Doctor A's $80\%$.
* **The Mathematical Reality:** Doctor B's aggregate success rate is heavily weighted by the fact that $90\%$ of their cases are "Easy" surgeries. Doctor A's overall rate is heavily dragged down because $90\%$ of their cases are the highly risky "Hard" surgeries.

---

## Real-World Applications

### A. University Admissions
A famous historical legal case involved a major university being sued for sex discrimination in its graduate admissions.

* **The Aggregated Data:** Overall, male applicants had a significantly higher admission rate than female applicants.
* **The Stratified Reality:** When conditioned on individual departments, women actually had higher admission rates in most cases. The paradox occurred because women applied in large numbers to highly competitive departments with low overall admission rates, while men applied to less competitive departments that were mathematically easier to get into.

### B. Baseball Batting Averages
It is entirely possible for Player A to have a higher batting average than Player B in the first half of the season *and* the second half of the season, yet possess a lower batting average for the entire season. This occurs if the number of "at-bats" (the weighting) varies drastically between the two players across the two halves.

---

## Mathematical Formulation

We can express this rigorously using conditional probability. Let:

* $S$ be the event of Success.
* $T$ be the Treatment/Group (e.g., Doctor B vs. Doctor A).
* $C$ be the Confounder (e.g., Surgery Type).

Simpson's Paradox occurs when the conditional probabilities point in one direction:
$$P(S \mid T, C) < P(S \mid T^c, C)$$
$$P(S \mid T, C^c) < P(S \mid T^c, C^c)$$

But the marginal (unconditioned) probabilities point in the exact opposite direction:
$$P(S \mid T) > P(S \mid T^c)$$

### Why the Law of Total Probability Doesn't Prevent This
The [Law of Total Probability](law-of-total-probability.ipynb) states:
$$P(S \mid T) = P(S \mid T, C)P(C \mid T) + P(S \mid T, C^c)P(C^c \mid T)$$

We cannot simply "add" or transitively apply the inequalities from the subgroups to the whole because of the weighting terms: $P(C \mid T)$ and $P(C^c \mid T)$. If these probability weights are vastly asymmetric (i.e., one doctor does mostly easy surgeries while the other does mostly hard ones), the overall average can swing radically, inverting the final inequality.

---

## Python Demonstration

We can use `pandas` to easily replicate and prove the paradox in a dataframe environment, showing how `.groupby()` yields drastically different results than an overall `.mean()`.

In [1]:
import pandas as pd

# Recreating the Surgeon data
data = {
    'Doctor': ['Doctor A']*100 + ['Doctor B']*100,
    'Surgery': ['Heart']*90 + ['Band-Aid']*10 + ['Heart']*10 + ['Band-Aid']*90,
    'Success': [1]*70 + [0]*20 + [1]*10 + [1]*2 + [0]*8 + [1]*81 + [0]*9
}

df = pd.DataFrame(data)
print("--- Data ---")
print(df.sample(10).head())
print("\n")


# 1. Aggregated Success Rate
print("--- Aggregated Overall Success Rate ---")
print(df.groupby('Doctor')['Success'].mean().round(3) * 100)
print("\n")

# 2. Conditioned Success Rate (By Surgery Type)
print("--- Stratified Success Rate (By Surgery Type) ---")
print(df.groupby(['Surgery', 'Doctor'])['Success'].mean().round(3) * 100)

--- Data ---
       Doctor   Surgery  Success
12   Doctor A     Heart        1
70   Doctor A     Heart        0
21   Doctor A     Heart        1
173  Doctor B  Band-Aid        1
40   Doctor A     Heart        1


--- Aggregated Overall Success Rate ---
Doctor
Doctor A    80.0
Doctor B    83.0
Name: Success, dtype: float64


--- Stratified Success Rate (By Surgery Type) ---
Surgery   Doctor  
Band-Aid  Doctor A    100.0
          Doctor B     90.0
Heart     Doctor A     77.8
          Doctor B     20.0
Name: Success, dtype: float64
